In [ ]:
# !pip install easyocr python-docx python-pptx scikit-learn pandas PyMuPDF

In [ ]:
import os
import re
import pandas as pd
import easyocr
import fitz  # PyMuPDF
from docx import Document
from pptx import Presentation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Initialize EasyOCR reader
reader = easyocr.Reader(['en'])

def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    
    try:
        if ext in ['.jpg', '.jpeg', '.png']:
            results = reader.readtext(file_path, detail=0)
            text = " ".join(results)
        
        elif ext == '.pdf':
            doc = fitz.open(file_path)
            for page in doc:
                text += page.get_text()
            # If PDF is scanned (no text), try OCR on first page
            if not text.strip():
                pix = doc[0].get_pixmap()
                results = reader.readtext(pix.tobytes(), detail=0)
                text = " ".join(results)
        
        elif ext == '.docx':
            doc = Document(file_path)
            text = " ".join([para.text for para in doc.paragraphs])
            
        elif ext == '.pptx':
            prs = Presentation(file_path)
            for slide in prs.slides:
                for shape in slide.shapes:
                    if hasattr(shape, "text"):
                        text += shape.text + " "
        return text.strip()
    except Exception as e:
        return f"Error processing {file_path}: {str(e)}"

def process_folder(folder_path):
    data = []
    supported_exts = ['.jpg', '.jpeg', '.png', '.pdf', '.docx', '.pptx']
    
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created input folder: {folder_path}. Please add files to it.")
        return pd.DataFrame()

    files = [f for f in os.listdir(folder_path) if os.path.splitext(f)[1].lower() in supported_exts]
    
    if not files:
        print(f"No supported files found in {folder_path}")
        return pd.DataFrame()

    for filename in files:
        file_path = os.path.join(folder_path, filename)
        content = extract_text(file_path)
        
        num_chars = len(content)
        num_ints = len(re.findall(r'\d+', content))
        extension = os.path.splitext(filename)[1]
        
        data.append({
            'file_name': filename,
            'extension': extension,
            'num_characters': num_chars,
            'num_integers': num_ints,
            'content': content
        })
    
    df = pd.DataFrame(data)
    
    # Intelligence: Grouping by similarity using KMeans
    if len(df) > 1:
        vectorizer = TfidfVectorizer(stop_words='english')
        X = vectorizer.fit_transform(df['content'].fillna(''))
        
        # Auto-determine clusters
        n_clusters = max(2, min(len(df), 5))
        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        df['group_id'] = kmeans.fit_predict(X)
    else:
        df['group_id'] = 0
        
    return df

# Configuration for input/output folders
input_folder = os.path.join(os.getcwd(), 'input')
output_folder = os.path.join(os.getcwd(), 'Output')

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Execution
results_df = process_folder(input_folder)

if not results_df.empty:
    output_path = os.path.join(output_folder, 'ocr_results.csv')
    results_df.to_csv(output_path, index=False)
    print(f"Processing complete. Results saved to {output_path}")
    display(results_df)